## Importing libraries

In [ ]:
import numpy as np
import pandas as pd

## Importing data

Columns to drop

In [ ]:
columns_to_not_use= [
    "TIPO_ATTIVITA_COD", # only `RI` values

    "IDNOSOLOGICO_PUERPERA",
    "INTERVENTO_ESTERNO_FLG",
    "'INTERVENTO_P'",
    "'INTERVENTO_S01'",
    "'INTERVENTO_S02'",
    "'INTERVENTO_S03'",
    "'INTERVENTO_S04'",
    "'INTERVENTO_S'"
]

Columns to format in a particular type

In [ ]:
sure_string_cols = [
    'PSEUDO_PAZ'
]
dtype = {col: 'string' for col in sure_string_cols}

string_cat_cols = [
    'MODALITA_DIMISSIONE_COD',
    'STRUTTURA_COD',
    'SESSO_COD',
    'PROVENIENZA_COD',
    'ISTRUZIONE_COD',
    'REGIME_RICOVERO_COD',
    'TIPO_RICOVERO_COD',
    'PRIORITA_PRENOTAZIONE_COD',
    'MOTIVO_RICOVERO_COD',
    'TRAUMA_INTOSSICAZIONE_COD',
    'TRAUMATISMO_CAUSA_COD',
    'RISCONTRO_AUTOPTICO',
    'MOTIVO_RICOVERO_DH_COD',
    'SDO_CHIUSURA',
    'SDO_RILEVAZIONE_DOLORE',
    'PROVENIENZA_RIAB_COD',
    'MODALITA_DIMISSIONE_RIAB_COD',
    'STRUTTURA_PROVENIENZA_COD',
    'STRUTTURA_DESTINAZIONE_COD',
    'DIAGNOSI_RICOVERO_FLG',
    'COMPILAZIONE_CHECKLIST_COD'
]
dtype.update({col: 'string' for col in string_cat_cols})

Import the excel file

In [ ]:
df = pd.read_excel('../raw_data/Estrazioni_iconsulting.xlsx', sheet_name='Ricoveri',
    usecols=lambda col: col not in columns_to_not_use,
    dtype_backend='numpy_nullable', # For allowing the `string` dtype
    dtype=dtype
)

# Make the index start from 2 (as the original excel)
df.index = df.index + 2

## Parsing and reformatting

Place NaT values instead of '9999-09-09 00:00:00' in columns:

In [ ]:
to_NaT_values = [
    'DATA_PRENOTAZIONE',
    'DATA_CHIUSURA_SDO'
]
df[to_NaT_values] = (df[to_NaT_values].replace(
    pd.Timestamp('9999-09-09 00:00:00'), pd.NaT
))

Place NA values instead of single whitespaces in columns:

In [ ]:
# to_NA_values = [
#     'PRIORITA_PRENOTAZIONE_COD',
#     'PROVENIENZA_RIAB_COD',
#     'MODALITA_DIMISSIONE_RIAB_COD',
#     'STRUTTURA_PROVENIENZA_COD',
#     'STRUTTURA_DESTINAZIONE_COD',
#     'DIAGNOSI_RICOVERO_FLG'
# ]
# df[to_NA_values] = (df[to_NA_values].replace(' ', pd.NA))

### Reformatting last column (`LISTAGG...`)

In [ ]:
ita_to_eng = {
    'GEN': 'JAN', 'FEB': 'FEB', 'MAR': 'MAR', 'APR': 'APR',
    'MAG': 'MAY', 'GIU': 'JUN', 'LUG': 'JUL', 'AGO': 'AUG',
    'SET': 'SEP', 'OTT': 'OCT', 'NOV': 'NOV', 'DIC': 'DEC'
}

def parse_italian_date(date_str):
    """Requires pandas. Convert italian date string like `31-GEN-99` to datetime"""
    
    date_str = str(date_str).strip()
    for ita, eng in ita_to_eng.items():
        date_str = date_str.replace(ita, eng)
    
    return pd.to_datetime(date_str, format='%d-%b-%y')

def split_column(value, ita_to_eng):
    """Requires pandas.
    Split a value made of comma separated dates in format like `31-GEN-99` into:
    - a column containing the dates in form of list of pandas datetime dates
    - a column containing the number of dates
    Place NaT and NA if value is null."""

    # Check if value is NA
    if pd.isna(value):
        return pd.Series([pd.NaT, pd.NA])

    # Place splitted date into a dictionary {date: frequency}
    dates_freq = {}
    for date in str(value).split(','):
        if date not in dates_freq:
            # dates_freq[parse_italian_date(date.strip())] = 1
            dates_freq[date.strip()] = 1
        else:
            # dates_freq[parse_italian_date(date.strip())] += 1
            dates_freq[date.strip()] += 1

    # Output is the dictionary itself and the number of total interventi
    return pd.Series([dates_freq, sum(dates_freq.values())])

In [ ]:
# Get position and split
original_col_name = "LISTAGG(C.DATA_INTERVENTO,\',\')OVER(PARTITIONBYA.NOSOLOGICO,A.ANNO,A.STRUTTURA_COD)"
original_position = df.columns.get_loc(original_col_name)

# Split the column
split_df = df[original_col_name].apply(split_column, args=(ita_to_eng,))
split_df.columns = ['DATE_INTERVENTI', 'NUM_INTERVENTI']
split_df['NUM_INTERVENTI'] = pd.to_numeric(split_df['NUM_INTERVENTI'], errors='coerce').astype('Int64')

# Reconstruct dataframe: columns before + new columns + columns after
df = pd.concat([
    df.iloc[:, :original_position
        # +1 # Decomment for keeping the original column
    ],
    split_df,
    df.iloc[:, original_position+1:]
], axis=1)

## Final checks

In [ ]:
df

In [ ]:
df.info()

## Export

In [ ]:
# # Parquet
# df.to_parquet("../raw_data/parquets/df-2_1-estr_icons-ricoveri.parquet")

# # Pickle
# df.to_pickle("../raw_data/pickles/df-2_1-estr_icons-ricoveri.pickle")

## Inconsistency between `PSEUDO_PAZ` and other columns

### Proof of the inconsistency: see results for code structure

`PROVENIENZA_COD`, `ISTRUZIONE_COD` are inconsistent

In [ ]:
print((df.groupby('PSEUDO_PAZ')['PROVENIENZA_COD'].nunique() == 1).all())
print((df.groupby('PSEUDO_PAZ')['ISTRUZIONE_COD'].nunique() == 1).all())

### See which `PSEUDO_PAZ` have multiple characteristics that should be unique

For `PROVENIENZA_COD` column

In [ ]:
incons = df.groupby('PSEUDO_PAZ')['PROVENIENZA_COD'].nunique()
incons = incons[incons > 1].sort_values(ascending=False)
incons

For `ISTRUZIONE_COD` column

In [ ]:
incons = df.groupby('PSEUDO_PAZ')['ISTRUZIONE_COD'].nunique()
incons = incons[incons > 1].sort_values(ascending=False)
incons